In [16]:
import pandas as pd
import requests
import os

url = "https://api.worldbank.org/v2/country/USA/indicator/EN.ATM.CO2E.PC?format=json&per_page=100"
headers = {'User-Agent': 'Mozilla/5.0'}

backup_data = [
    {"date": "2020", "value": 13.03, "countryiso3code": "USA", "indicator": {"id": "EN.ATM.CO2E.PC"}},
    {"date": "2019", "value": 14.67, "countryiso3code": "USA", "indicator": {"id": "EN.ATM.CO2E.PC"}},
    {"date": "2018", "value": 15.24, "countryiso3code": "USA", "indicator": {"id": "EN.ATM.CO2E.PC"}},
    {"date": "2017", "value": 14.82, "countryiso3code": "USA", "indicator": {"id": "EN.ATM.CO2E.PC"}}
]

try:
    print("Запрос к API...")
    response = requests.get(url, headers=headers)
    raw_data = response.json()

    if isinstance(raw_data, list) and len(raw_data) > 1:
        df = pd.json_normalize(raw_data[1])
        print("Успех: Данные получены из API!")

    else:
        if isinstance(raw_data, list) and len(raw_data) > 0 and 'message' in raw_data[0]:
            print("API недоступен:", raw_data[0]['message'][0]['value'])

        print("Используем План Б (резервные данные)...")
        df = pd.json_normalize(backup_data)

    display(df.head())

except Exception as e:
    print(f"Системная ошибка: {e}")
    print("Используем План Б (резервные данные)...")
    df = pd.json_normalize(backup_data)
    display(df.head())

Запрос к API...
API недоступен: The indicator was not found. It may have been deleted or archived.
Используем План Б (резервные данные)...


,date,value,countryiso3code,indicator.id
0,2020,13.03,USA,EN.ATM.CO2E.PC
1,2019,14.67,USA,EN.ATM.CO2E.PC
2,2018,15.24,USA,EN.ATM.CO2E.PC
3,2017,14.82,USA,EN.ATM.CO2E.PC


In [17]:
cols_to_keep = ['date', 'value', 'countryiso3code']
if 'indicator.id' in df.columns:
    df_clean = df[cols_to_keep + ['indicator.id']].copy()
    df_clean.rename(columns={'indicator.id': 'indicator'}, inplace=True)
else:
    df_clean = df[cols_to_keep + ['indicator']].copy()

df_clean.rename(columns={'date': 'year'}, inplace=True)

df_clean['year'] = df_clean['year'].astype(int)
df_clean['value'] = pd.to_numeric(df_clean['value'])

df_clean = df_clean.dropna(subset=['value'])

print("Данные очищены!")
display(df_clean.head())

Данные очищены!


,year,value,countryiso3code,indicator
0,2020,13.03,USA,EN.ATM.CO2E.PC
1,2019,14.67,USA,EN.ATM.CO2E.PC
2,2018,15.24,USA,EN.ATM.CO2E.PC
3,2017,14.82,USA,EN.ATM.CO2E.PC


In [18]:
import os
from datetime import datetime

path = "../data/normalized/variant_14/"
os.makedirs(path, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"usa_co2_normalized_{timestamp}.csv"

df_clean.to_csv(os.path.join(path, filename), index=False)

print(f"Файл успешно сохранен: {path}{filename}")

Файл успешно сохранен: ../data/normalized/variant_14/usa_co2_normalized_20260318_1402.csv
